# 🌱 FarmTech Solutions — Análise de Rendimento Agrícola

## Fase 5 — Machine Learning

### Integrantes

- João Vitor Justino — RM572969
- Thiese Barros de Novaes — RM572659
- Renan de Souza Silva — RM568958
- Talles Duran das Virgens — RM572772
- Kevin Souza Santiago — RM573808

---

## Objetivo do projeto

Este notebook tem como objetivo analisar a base `crop_yield.csv` e desenvolver uma solução de Machine Learning capaz de **prever o rendimento agrícola (`Yield`)** a partir de informações ambientais.

As variáveis disponíveis incluem:

- cultura agrícola (`Crop`);
- precipitação;
- umidade específica;
- umidade relativa;
- temperatura;
- rendimento da safra (`Yield`).

O trabalho será desenvolvido em etapas. Primeiro será realizada a compreensão e validação da base. Em seguida serão feitas a análise exploratória, identificação de padrões, investigação de possíveis valores discrepantes, clusterização e, posteriormente, comparação de diferentes modelos de regressão supervisionada.

### Perguntas que orientam a análise

1. Como as condições ambientais se distribuem na base?
2. As culturas apresentam comportamentos de rendimento diferentes?
3. Existem padrões ou grupos naturais entre as observações?
4. Existem registros potencialmente discrepantes?
5. Quais variáveis ajudam a explicar o rendimento?
6. Qual modelo de regressão apresenta melhor capacidade preditiva?

> **Importante:** nenhuma conclusão será assumida antecipadamente. Cada interpretação será baseada nos resultados obtidos durante a análise.


## 1. Preparação do ambiente

Nesta etapa são importadas as bibliotecas necessárias para manipulação, análise e visualização dos dados.

- **Pandas:** manipulação de tabelas e análise de dados;
- **NumPy:** operações numéricas;
- **Matplotlib e Seaborn:** construção de visualizações;
- **Pathlib:** manipulação de caminhos de arquivos de forma independente do sistema operacional.

Também definimos um `RANDOM_STATE` fixo. Ele será utilizado posteriormente nos algoritmos de Machine Learning para tornar os resultados reproduzíveis.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("✅ Ambiente preparado com sucesso.")


## 2. Carregamento e validação da base

A base utilizada é o arquivo `crop_yield.csv`.

Antes de iniciar qualquer análise estatística, precisamos garantir que:

- o arquivo foi encontrado;
- a leitura foi realizada corretamente;
- as colunas esperadas estão presentes;
- o conjunto de dados não está vazio.

Essa validação evita que etapas posteriores sejam executadas sobre um arquivo incorreto ou com estrutura diferente da esperada.


In [ ]:
def localizar_dataset(nome_arquivo="crop_yield.csv"):
    # Procura o dataset tanto a partir da raiz do projeto quanto da pasta notebook.
    candidatos = [
        Path.cwd() / "data" / nome_arquivo,
        Path.cwd().parent / "data" / nome_arquivo,
        Path.cwd() / nome_arquivo,
    ]

    for caminho in candidatos:
        caminho = caminho.resolve()
        if caminho.exists():
            return caminho

    caminhos_testados = "\n".join(f"- {c.resolve()}" for c in candidatos)
    raise FileNotFoundError(
        f"Dataset '{nome_arquivo}' não encontrado.\n"
        f"Caminhos verificados:\n{caminhos_testados}"
    )


CAMINHO_DADOS = localizar_dataset()

df = pd.read_csv(CAMINHO_DADOS)

COLUNAS_ESPERADAS = [
    "Crop",
    "Precipitation (mm day-1)",
    "Specific Humidity at 2 Meters (g/kg)",
    "Relative Humidity at 2 Meters (%)",
    "Temperature at 2 Meters (C)",
    "Yield",
]

colunas_ausentes = [col for col in COLUNAS_ESPERADAS if col not in df.columns]

if colunas_ausentes:
    raise ValueError(f"Colunas esperadas não encontradas: {colunas_ausentes}")

if df.empty:
    raise ValueError("O dataset foi carregado, mas está vazio.")

print(f"✅ Dataset encontrado: {CAMINHO_DADOS}")
print(f"✅ Registros: {df.shape[0]}")
print(f"✅ Variáveis: {df.shape[1]}")
print("✅ Estrutura básica validada.")


### 2.1 Primeiras observações

A visualização das primeiras linhas serve para confirmar se os dados foram interpretados corretamente pelo Pandas e para conhecer o formato das variáveis.

Neste momento não serão feitas conclusões sobre relações entre as variáveis. O objetivo é apenas compreender a estrutura inicial da base.


In [ ]:
display(df.head())


### 2.2 Dicionário das variáveis

| Variável | Tipo esperado | Papel na análise |
|---|---|---|
| `Crop` | Categórica | Identifica a cultura agrícola |
| `Precipitation (mm day-1)` | Numérica | Informação de precipitação |
| `Specific Humidity at 2 Meters (g/kg)` | Numérica | Umidade específica do ar |
| `Relative Humidity at 2 Meters (%)` | Numérica | Umidade relativa do ar |
| `Temperature at 2 Meters (C)` | Numérica | Temperatura a 2 metros |
| `Yield` | Numérica | Variável-alvo de rendimento |

A variável que desejamos prever posteriormente é **`Yield`**.

> A unidade e os valores serão preservados exatamente como fornecidos no dataset. Caso alguma faixa pareça incomum, ela será investigada antes de qualquer alteração.


### 2.3 Estrutura técnica da base

Agora verificamos tipos de dados, quantidade de valores não nulos, quantidade de nulos e número de valores distintos.

Essa visão é mais útil do que observar apenas `df.info()`, pois transforma a estrutura da base em uma tabela comparável.


In [ ]:
estrutura = pd.DataFrame({
    "Tipo": df.dtypes.astype(str),
    "Não nulos": df.notna().sum(),
    "Nulos": df.isna().sum(),
    "Valores únicos": df.nunique()
})

display(estrutura)

print(f"\nDimensão da base: {df.shape[0]} linhas × {df.shape[1]} colunas")


### Interpretação da estrutura

A base possui **156 registros e 6 variáveis**.

A coluna `Crop` é categórica, enquanto as outras cinco variáveis são numéricas. Isso é coerente com o problema proposto: a cultura representa uma categoria e as demais colunas descrevem condições ambientais ou rendimento.

A variável-alvo `Yield` está armazenada como valor numérico, permitindo sua utilização em modelos de regressão.

Antes de treinar modelos, a variável categórica `Crop` precisará ser tratada de forma apropriada para que os algoritmos consigam utilizá-la.


## 3. Verificação da qualidade dos dados

Antes da análise estatística e do Machine Learning, verificamos problemas que podem comprometer os resultados:

- valores ausentes;
- registros duplicados;
- valores infinitos;
- valores fisicamente inválidos em verificações básicas.

Essa etapa evita que erros de qualidade sejam confundidos com padrões reais dos dados.


In [ ]:
nulos = pd.DataFrame({
    "Valores ausentes": df.isna().sum(),
    "Percentual (%)": (df.isna().mean() * 100).round(2)
})

duplicados = int(df.duplicated().sum())

colunas_numericas = df.select_dtypes(include=np.number).columns
infinitos = int(np.isinf(df[colunas_numericas]).sum().sum())

validacoes_basicas = {
    "Precipitação negativa": int((df["Precipitation (mm day-1)"] < 0).sum()),
    "Umidade específica negativa": int((df["Specific Humidity at 2 Meters (g/kg)"] < 0).sum()),
    "Umidade relativa fora de 0–100%": int(
        (~df["Relative Humidity at 2 Meters (%)"].between(0, 100)).sum()
    ),
    "Yield menor ou igual a zero": int((df["Yield"] <= 0).sum()),
}

display(nulos)

print(f"Registros duplicados: {duplicados}")
print(f"Valores infinitos em colunas numéricas: {infinitos}")

print("\nVerificações básicas:")
for regra, quantidade in validacoes_basicas.items():
    print(f"- {regra}: {quantidade}")


### Interpretação da qualidade da base

A base não apresenta valores ausentes nem registros duplicados. Também não foram identificados valores infinitos nas variáveis numéricas.

Nas verificações físicas básicas não aparecem precipitações negativas, umidade específica negativa, umidade relativa fora da faixa de 0% a 100% ou rendimentos menores ou iguais a zero.

Portanto, **não há necessidade de imputar valores ausentes ou remover duplicidades nesta etapa**.

Isso não significa que a base esteja automaticamente livre de valores discrepantes. A análise de possíveis outliers será feita posteriormente levando em consideração o comportamento de cada cultura.


## 4. Análise estatística descritiva

A estatística descritiva permite compreender a escala e a dispersão das variáveis numéricas antes das visualizações e dos modelos.

Serão observados:

- média;
- desvio padrão;
- mínimo;
- quartis;
- mediana;
- máximo.

É importante comparar **média e mediana**, pois diferenças grandes entre elas podem indicar assimetria na distribuição ou mistura de grupos com comportamentos distintos.


In [ ]:
estatisticas = df.describe().T.round(2)

display(estatisticas)


### Interpretação das estatísticas numéricas

As variáveis ambientais apresentam faixas relativamente concentradas:

- a temperatura varia de aproximadamente **25,56 °C a 26,81 °C**;
- a umidade relativa varia de aproximadamente **82,11% a 86,10%**;
- a umidade específica varia de aproximadamente **17,54 a 18,70 g/kg**;
- a precipitação varia de aproximadamente **1.934,62 a 3.085,79**, mantendo-se a unidade apresentada pela própria base.

O `Yield`, por outro lado, apresenta dispersão muito maior:

- mínimo: **5.249**;
- mediana: **18.871**;
- média: aproximadamente **56.153**;
- máximo: **203.399**.

A média de `Yield` é muito superior à mediana. Isso sugere que a distribuição geral do rendimento não é simétrica.

Porém, **ainda não é correto concluir que os maiores valores são outliers**. Como a base contém diferentes culturas, elas podem possuir níveis naturais de produtividade muito diferentes. Por isso, o próximo diagnóstico descritivo deve separar o rendimento por cultura.


### 4.1 Distribuição das culturas e rendimento por grupo

Antes de procurar valores discrepantes, verificamos quantas observações existem de cada cultura e como o rendimento se comporta em cada grupo.

Isso é importante porque um valor elevado para uma cultura pode ser perfeitamente normal para ela, mesmo que pareça extremo quando todas as culturas são analisadas juntas.


In [ ]:
contagem_culturas = (
    df["Crop"]
    .value_counts()
    .rename_axis("Cultura")
    .reset_index(name="Quantidade")
)

resumo_yield_cultura = (
    df.groupby("Crop")["Yield"]
    .agg(
        quantidade="count",
        media="mean",
        mediana="median",
        minimo="min",
        maximo="max",
        desvio_padrao="std"
    )
    .round(2)
    .sort_values("media", ascending=False)
)

print(f"Quantidade de culturas diferentes: {df['Crop'].nunique()}\n")
display(contagem_culturas)
display(resumo_yield_cultura)


### Interpretação por cultura

A base contém **quatro culturas**, com **39 observações em cada grupo**:

- `Cocoa, beans`;
- `Oil palm fruit`;
- `Rice, paddy`;
- `Rubber, natural`.

Essa distribuição perfeitamente equilibrada é favorável para comparações entre culturas, pois nenhum grupo domina numericamente o conjunto de dados.

O rendimento, entretanto, varia fortemente entre as culturas. A média de `Oil palm fruit` é muito superior às demais, enquanto `Cocoa, beans` e `Rubber, natural` apresentam rendimentos médios consideravelmente menores.

Isso explica grande parte da dispersão observada no `Yield` quando todas as culturas são analisadas juntas.

**Conclusão desta etapa:** valores altos de rendimento não devem ser tratados automaticamente como outliers. A identificação de valores discrepantes precisa considerar a cultura à qual cada observação pertence.


## ✅ Conclusão parcial — preparação concluída

Até este ponto foi possível confirmar que:

- o dataset foi carregado e validado corretamente;
- a base possui 156 registros e 6 variáveis;
- existem quatro culturas, com 39 observações cada;
- não há valores ausentes;
- não há registros duplicados;
- não foram encontrados valores infinitos;
- as verificações físicas básicas não indicaram inconsistências óbvias;
- `Yield` apresenta grande dispersão;
- parte importante dessa dispersão está relacionada às diferenças entre culturas.

### Atenção para a próxima etapa

Os valores de precipitação parecem numericamente elevados para uma interpretação literal de `mm day-1`. Como a base e o enunciado fornecem essa nomenclatura, **não alteraremos a unidade ou os valores sem evidência adicional**. A informação será preservada e tratada como uma característica do dataset.

A próxima etapa será a **Análise Exploratória dos Dados (EDA)**, utilizando visualizações para investigar distribuições, relações entre variáveis e possíveis valores discrepantes.


# 5. Análise Exploratória dos Dados (EDA)

A Análise Exploratória dos Dados tem como objetivo compreender visualmente e estatisticamente os padrões presentes na base antes da aplicação dos algoritmos de Machine Learning.

Nesta etapa serão investigados:

- equilíbrio entre as culturas;
- distribuição das variáveis numéricas;
- comportamento do rendimento (`Yield`);
- diferenças de rendimento entre culturas;
- estrutura dos perfis ambientais;
- relações entre variáveis ambientais;
- correlações globais e por cultura.

A EDA é importante porque ajuda a identificar características da base que podem influenciar tanto a clusterização quanto os modelos de regressão.


## 5.1 Distribuição das culturas

Primeiro verificamos visualmente se as culturas possuem quantidades semelhantes de observações.

Uma distribuição muito desequilibrada poderia fazer com que determinadas culturas tivessem influência excessiva nas análises posteriores.


In [ ]:
plt.figure(figsize=(10, 5))

sns.countplot(
    data=df,
    x="Crop",
    order=df["Crop"].value_counts().index
)

plt.title("Quantidade de Registros por Cultura")
plt.xlabel("Cultura")
plt.ylabel("Quantidade de registros")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


### Interpretação

As quatro culturas possuem **39 observações cada**, totalizando 156 registros.

Portanto, a base está perfeitamente equilibrada em relação à variável `Crop`.

Esse equilíbrio favorece comparações entre as culturas, pois nenhuma delas possui maior representação numérica no conjunto de dados.


## 5.2 Distribuição das variáveis numéricas

Agora analisamos individualmente a distribuição de cada variável numérica.

Os histogramas ajudam a observar:

- concentração dos valores;
- assimetria;
- amplitude;
- possíveis regiões com baixa ou alta frequência.

Neste momento, os gráficos são utilizados para compreender o formato das distribuições, e não para remover observações.


In [ ]:
variaveis_numericas = [
    "Precipitation (mm day-1)",
    "Specific Humidity at 2 Meters (g/kg)",
    "Relative Humidity at 2 Meters (%)",
    "Temperature at 2 Meters (C)",
    "Yield",
]

for coluna in variaveis_numericas:
    plt.figure(figsize=(9, 5))
    sns.histplot(data=df, x=coluna, bins=15, kde=True)
    plt.title(f"Distribuição — {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.show()


### Interpretação das distribuições

As variáveis ambientais apresentam faixas relativamente concentradas, principalmente temperatura e umidades.

A variável `Yield` apresenta comportamento diferente: sua distribuição é claramente mais assimétrica e possui uma amplitude muito maior.

Essa assimetria não deve ser interpretada imediatamente como presença de erros ou outliers, pois o conjunto contém culturas com níveis de rendimento naturalmente muito diferentes.

Por esse motivo, o próximo passo é analisar `Yield` separadamente por cultura.


## 5.3 Distribuição do rendimento por cultura

O rendimento geral mistura quatro culturas diferentes. Para evitar conclusões incorretas, analisamos sua distribuição de forma separada.

O boxplot permite comparar:

- mediana;
- dispersão;
- intervalo interquartil;
- diferenças de escala;
- possíveis observações extremas dentro de cada cultura.


In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="Crop",
    y="Yield"
)

plt.title("Distribuição do Rendimento por Cultura")
plt.xlabel("Cultura")
plt.ylabel("Yield")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
resumo_yield_eda = (
    df.groupby("Crop")["Yield"]
    .agg(
        media="mean",
        mediana="median",
        desvio_padrao="std",
        minimo="min",
        maximo="max"
    )
    .round(2)
    .sort_values("media", ascending=False)
)

display(resumo_yield_eda)


### Interpretação

A análise confirma que a cultura é uma variável extremamente importante para compreender o rendimento.

Os rendimentos médios são aproximadamente:

- **Oil palm fruit:** 175,8 mil;
- **Rice, paddy:** 32,1 mil;
- **Cocoa, beans:** 8,9 mil;
- **Rubber, natural:** 7,8 mil.

`Oil palm fruit` apresenta uma escala de rendimento muito superior às demais culturas. Isso explica por que a média geral de `Yield` ficou muito acima da mediana na estatística descritiva.

Portanto, um valor alto de `Yield` não deve ser considerado discrepante apenas por estar distante da média global. A análise de outliers precisará ser feita considerando cada cultura separadamente.


## 5.4 Estrutura dos perfis ambientais

Antes de analisar correlações, verificamos se as condições ambientais são diferentes entre as culturas.

Essa verificação é relevante porque pode revelar como o dataset foi estruturado e evitar interpretações incorretas sobre diferenças climáticas entre os grupos.


In [ ]:
colunas_ambientais = [
    "Precipitation (mm day-1)",
    "Specific Humidity at 2 Meters (g/kg)",
    "Relative Humidity at 2 Meters (%)",
    "Temperature at 2 Meters (C)",
]

resumo_ambiental_por_cultura = (
    df.groupby("Crop")[colunas_ambientais]
    .mean()
    .round(3)
)

display(resumo_ambiental_por_cultura)

perfis_ambientais = (
    df.groupby(colunas_ambientais)
    .size()
    .reset_index(name="Quantidade_de_registros")
)

print(f"Perfis ambientais únicos: {len(perfis_ambientais)}")
print(
    "Quantidade de vezes que cada perfil aparece:",
    sorted(perfis_ambientais["Quantidade_de_registros"].unique())
)


### Interpretação da estrutura ambiental

Foi identificado um padrão importante na construção da base:

- existem **39 combinações únicas de condições ambientais**;
- cada combinação aparece exatamente **4 vezes**;
- isso corresponde às quatro culturas presentes na base.

Além disso, as médias de precipitação, umidade e temperatura são iguais entre as culturas.

Isso indica que as culturas foram avaliadas sob os **mesmos perfis ambientais**.

Essa característica é importante porque permite comparar como culturas diferentes respondem às mesmas condições, mas também deve ser considerada posteriormente na validação dos modelos de regressão.


## 5.5 Matriz de correlação global

A correlação de Pearson mede a intensidade da relação linear entre duas variáveis numéricas.

Os valores variam de:

- **+1:** relação linear positiva perfeita;
- **0:** ausência de relação linear;
- **-1:** relação linear negativa perfeita.

A correlação não implica causalidade. Ela será utilizada apenas como ferramenta exploratória.


In [ ]:
correlacao_global = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    correlacao_global,
    annot=True,
    fmt=".2f"
)

plt.title("Matriz de Correlação — Variáveis Numéricas")
plt.tight_layout()
plt.show()

display(correlacao_global.round(3))


### Interpretação da correlação global

Entre as variáveis ambientais, algumas relações lineares se destacam:

- precipitação × umidade relativa: correlação de aproximadamente **0,75**;
- temperatura × umidade específica: aproximadamente **0,70**;
- precipitação × umidade específica: aproximadamente **0,49**;
- temperatura × umidade relativa: aproximadamente **-0,34**.

Por outro lado, quando todas as culturas são analisadas juntas, as correlações lineares entre `Yield` e as variáveis ambientais ficam próximas de zero.

Isso **não significa necessariamente que clima e rendimento não tenham relação**.

Como cada cultura possui uma escala de rendimento muito diferente, misturar todos os grupos em uma única correlação pode ocultar relações existentes dentro de cada cultura. Por isso, a análise será repetida separadamente por grupo.


## 5.6 Correlação entre as variáveis ambientais e o rendimento por cultura

Agora calculamos a correlação de cada variável ambiental com `Yield` separadamente para cada cultura.

Essa abordagem permite verificar se a relação entre ambiente e rendimento muda dependendo da cultura analisada.


In [ ]:
correlacoes_por_cultura = {}

for cultura, grupo in df.groupby("Crop"):
    correlacoes = (
        grupo[colunas_ambientais + ["Yield"]]
        .corr()["Yield"]
        .drop("Yield")
    )
    correlacoes_por_cultura[cultura] = correlacoes

tabela_correlacoes_cultura = (
    pd.DataFrame(correlacoes_por_cultura)
    .T
    .round(3)
)

display(tabela_correlacoes_cultura)


### Interpretação das correlações por cultura

A análise separada revela relações que ficaram escondidas na correlação global.

#### Rice, paddy

Para o arroz, aparecem as relações lineares mais fortes com o rendimento:

- umidade específica × `Yield`: aproximadamente **0,70**;
- temperatura × `Yield`: aproximadamente **0,61**;
- precipitação × `Yield`: aproximadamente **0,33**.

Isso indica que, dentro desse grupo, condições ambientais apresentam associação linear mais evidente com o rendimento.

#### Rubber, natural

Para a borracha natural, as relações mais relevantes são negativas:

- umidade específica × `Yield`: aproximadamente **-0,43**;
- temperatura × `Yield`: aproximadamente **-0,41**.

#### Cocoa, beans e Oil palm fruit

Nessas culturas, as correlações lineares entre as variáveis ambientais e `Yield` são mais fracas na base analisada.

### Conclusão

A relação entre ambiente e rendimento **não é igual para todas as culturas**.

Esse resultado reforça a importância de incluir `Crop` nos modelos preditivos e mostra que uma análise exclusivamente global poderia esconder comportamentos específicos de cada cultura.


## 5.7 Relação visual entre variáveis ambientais e Yield

Para complementar os coeficientes de correlação, utilizamos gráficos de dispersão.

Cada ponto representa uma observação e as culturas são diferenciadas no gráfico. O objetivo é verificar visualmente se existem tendências, agrupamentos ou relações não lineares.


In [ ]:
for coluna in colunas_ambientais:
    plt.figure(figsize=(9, 6))

    sns.scatterplot(
        data=df,
        x=coluna,
        y="Yield",
        hue="Crop"
    )

    plt.title(f"{coluna} × Yield")
    plt.xlabel(coluna)
    plt.ylabel("Yield")
    plt.tight_layout()
    plt.show()


### Interpretação dos gráficos de dispersão

Os gráficos deixam evidente a separação das culturas em diferentes faixas de rendimento, principalmente para `Oil palm fruit`.

Também é possível perceber que as mesmas condições ambientais aparecem associadas a rendimentos muito diferentes dependendo da cultura.

Isso confirma visualmente dois pontos importantes:

1. `Crop` carrega informação essencial para prever `Yield`;
2. as relações entre ambiente e rendimento podem variar de uma cultura para outra.

Essas características deverão ser consideradas tanto na clusterização quanto nos modelos supervisionados.


## ✅ Conclusão parcial da EDA

A análise exploratória permitiu identificar os seguintes pontos:

- as quatro culturas possuem a mesma quantidade de observações;
- as variáveis ambientais apresentam faixas relativamente concentradas;
- `Yield` possui grande dispersão quando todas as culturas são analisadas juntas;
- essa dispersão é explicada principalmente pelas diferenças de rendimento entre as culturas;
- `Oil palm fruit` apresenta rendimento muito superior aos demais grupos;
- existem 39 perfis ambientais únicos, cada um repetido uma vez para cada cultura;
- as culturas foram observadas sob os mesmos perfis ambientais;
- a correlação global de `Yield` com as variáveis ambientais é baixa;
- entretanto, análises por cultura revelam relações mais fortes, especialmente para `Rice, paddy` e `Rubber, natural`;
- a variável `Crop` deverá ter papel importante na modelagem preditiva.

A próxima etapa será a **identificação de possíveis valores discrepantes (outliers)**. Essa análise será realizada por cultura para evitar classificar como anormal um rendimento que apenas pertence a uma cultura naturalmente mais produtiva.


# 6. Identificação e análise de outliers

Após a EDA, o próximo passo é investigar possíveis valores discrepantes.

Um **outlier** é uma observação que se encontra distante da maior parte dos dados. Entretanto, um valor estatisticamente extremo não é necessariamente um erro.

Neste projeto isso é especialmente importante porque:

- as culturas possuem níveis de rendimento muito diferentes;
- as variáveis ambientais apresentam faixas estreitas;
- os mesmos 39 perfis ambientais aparecem para todas as quatro culturas.

Por esse motivo, nenhum registro será removido automaticamente. Primeiro será feita a detecção estatística e, depois, a interpretação do resultado.


## 6.1 Método utilizado — Intervalo Interquartil (IQR)

Será utilizado o método do **Intervalo Interquartil (IQR)**.

O cálculo é:

- `Q1`: primeiro quartil, correspondente ao percentil 25%;
- `Q3`: terceiro quartil, correspondente ao percentil 75%;
- `IQR = Q3 - Q1`.

Os limites normalmente utilizados são:

`Limite inferior = Q1 - 1,5 × IQR`

`Limite superior = Q3 + 1,5 × IQR`

Valores abaixo do limite inferior ou acima do limite superior são sinalizados como **potenciais outliers**.

> O termo "potencial" é importante: o método identifica pontos estatisticamente extremos, mas a decisão de remover ou manter cada observação depende do contexto do problema.


In [ ]:
def limites_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    return q1, q3, iqr, limite_inferior, limite_superior


def identificar_outliers_iqr(serie):
    q1, q3, iqr, limite_inferior, limite_superior = limites_iqr(serie)

    mascara = (serie < limite_inferior) | (serie > limite_superior)

    return mascara, limite_inferior, limite_superior


## 6.2 O risco de detectar outliers de `Yield` globalmente

Primeiro aplicamos o IQR em `Yield` sem separar as culturas.

Esse teste é proposital: ele mostra por que uma aplicação mecânica do método pode gerar uma conclusão incorreta neste dataset.


In [ ]:
mascara_global_yield, limite_inf_yield, limite_sup_yield = identificar_outliers_iqr(
    df["Yield"]
)

quantidade_global_yield = int(mascara_global_yield.sum())

print(f"Limite inferior global: {limite_inf_yield:,.2f}")
print(f"Limite superior global: {limite_sup_yield:,.2f}")
print(f"Potenciais outliers globais de Yield: {quantidade_global_yield}")


In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x="Yield"
)

plt.title("Boxplot Global do Rendimento")
plt.xlabel("Yield")
plt.tight_layout()
plt.show()


### Interpretação

Quando todas as culturas são misturadas, o método IQR sinaliza **35 observações de `Yield` como potenciais outliers**.

Entretanto, a EDA mostrou que `Oil palm fruit` possui uma escala de rendimento muito superior às demais culturas.

Portanto, esses valores não devem ser removidos simplesmente porque estão acima do limite global.

Nesse caso, a análise global gera **falsos outliers contextuais**: valores que parecem extremos quando comparados com todas as culturas, mas podem ser completamente normais dentro de sua própria cultura.


## 6.3 Detecção de outliers de `Yield` por cultura

Para evitar o problema anterior, o IQR será calculado separadamente dentro de cada cultura.

Essa abordagem compara cada observação apenas com registros do mesmo grupo agrícola.


In [ ]:
resumo_outliers_yield = []

for cultura, grupo in df.groupby("Crop"):
    mascara, limite_inferior, limite_superior = identificar_outliers_iqr(
        grupo["Yield"]
    )

    q1, q3, iqr, _, _ = limites_iqr(grupo["Yield"])

    resumo_outliers_yield.append({
        "Cultura": cultura,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Limite inferior": limite_inferior,
        "Limite superior": limite_superior,
        "Outliers": int(mascara.sum())
    })

resumo_outliers_yield = pd.DataFrame(resumo_outliers_yield)

display(resumo_outliers_yield.round(2))


In [ ]:
plt.figure(figsize=(11, 6))

sns.boxplot(
    data=df,
    x="Crop",
    y="Yield"
)

plt.title("Outliers de Yield Avaliados por Cultura")
plt.xlabel("Cultura")
plt.ylabel("Yield")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


### Resultado e interpretação

Ao aplicar o IQR separadamente para cada cultura, o resultado é:

- `Cocoa, beans`: **0 outliers de Yield**;
- `Oil palm fruit`: **0 outliers de Yield**;
- `Rice, paddy`: **0 outliers de Yield**;
- `Rubber, natural`: **0 outliers de Yield**.

Isso confirma que os rendimentos elevados de `Oil palm fruit` representam o comportamento normal desse grupo e não observações isoladas anormais.

### Decisão

**Nenhuma observação de `Yield` será removida.**

Essa decisão preserva os 156 registros originais e evita eliminar informações legítimas de uma cultura apenas por ela possuir uma escala de rendimento diferente.


## 6.4 Análise de possíveis outliers nas variáveis ambientais

Agora aplicamos o mesmo método às variáveis ambientais.

Como essas variáveis possuem faixas bastante estreitas, o IQR pode sinalizar alguns valores próximos das extremidades mesmo quando eles são plausíveis.

Por isso, novamente faremos distinção entre **sinalização estatística** e **erro de dados**.


In [ ]:
colunas_ambientais = [
    "Precipitation (mm day-1)",
    "Specific Humidity at 2 Meters (g/kg)",
    "Relative Humidity at 2 Meters (%)",
    "Temperature at 2 Meters (C)",
]

resumo_outliers_ambientais = []

for coluna in colunas_ambientais:
    mascara, limite_inferior, limite_superior = identificar_outliers_iqr(
        df[coluna]
    )

    resumo_outliers_ambientais.append({
        "Variável": coluna,
        "Mínimo observado": df[coluna].min(),
        "Máximo observado": df[coluna].max(),
        "Limite inferior IQR": limite_inferior,
        "Limite superior IQR": limite_superior,
        "Registros sinalizados": int(mascara.sum())
    })

resumo_outliers_ambientais = pd.DataFrame(resumo_outliers_ambientais)

display(resumo_outliers_ambientais.round(3))


In [ ]:
for coluna in colunas_ambientais:
    plt.figure(figsize=(9, 4))
    sns.boxplot(data=df, x=coluna)
    plt.title(f"Boxplot — {coluna}")
    plt.tight_layout()
    plt.show()


### Interpretação das variáveis ambientais

A precipitação não apresenta observações sinalizadas pelo IQR global.

A temperatura apresenta alguns pontos estatisticamente extremos, e as umidades podem apresentar valores muito próximos dos limites quando analisadas por grupos menores.

Entretanto, isso não é evidência suficiente para considerar esses registros inválidos.

Por exemplo, a temperatura observada varia apenas de aproximadamente **25,56 °C a 26,81 °C**. Portanto, mesmo os valores nas extremidades continuam muito próximos da faixa central e são plausíveis dentro da própria base.

Além disso, os mesmos perfis ambientais são repetidos para as quatro culturas, indicando que esses valores fazem parte da estrutura planejada do dataset, e não de erros isolados de digitação ou medição.


## 6.5 Verificação por cultura das variáveis ambientais

Como confirmação adicional, contamos quantos pontos o IQR sinaliza dentro de cada cultura.

Essa análise serve principalmente para demonstrar que uma regra estatística deve ser interpretada com cautela quando a amostra possui pouca variabilidade.


In [ ]:
contagem_outliers_por_cultura = []

for cultura, grupo in df.groupby("Crop"):
    registro = {"Cultura": cultura}

    for coluna in colunas_ambientais:
        mascara, _, _ = identificar_outliers_iqr(grupo[coluna])
        registro[coluna] = int(mascara.sum())

    contagem_outliers_por_cultura.append(registro)

contagem_outliers_por_cultura = pd.DataFrame(
    contagem_outliers_por_cultura
).set_index("Cultura")

display(contagem_outliers_por_cultura)


### Interpretação

O IQR pode sinalizar alguns extremos nas variáveis de umidade e temperatura quando cada cultura é analisada isoladamente.

Isso ocorre porque as distribuições ambientais são muito concentradas: pequenas diferenças em relação aos quartis podem ultrapassar matematicamente o limite de `1,5 × IQR`.

Porém:

- os valores continuam dentro das faixas observadas da própria base;
- os mesmos extremos se repetem entre as quatro culturas;
- não há evidência de erro de coleta, valor impossível ou inconsistência estrutural.

Assim, **esses registros também serão mantidos**.


## 6.6 Decisão final sobre outliers

Após combinar a análise estatística com o contexto do dataset, foi tomada a seguinte decisão:

### `Yield`

- o IQR global sinaliza 35 potenciais outliers;
- essa sinalização é causada principalmente pelas diferenças estruturais entre culturas;
- ao calcular o IQR por cultura, nenhum `Yield` é classificado como outlier.

### Variáveis ambientais

- alguns valores nas extremidades podem ser sinalizados pelo IQR;
- as faixas continuam plausíveis;
- os mesmos perfis ambientais aparecem repetidamente nas quatro culturas;
- não existem evidências suficientes para justificar remoção.

### Decisão metodológica

**Nenhum registro será removido da base nesta etapa.**

A base continuará com os **156 registros originais**.

Essa decisão evita perda desnecessária de informação e preserva padrões que poderão ser importantes para a clusterização e para os modelos de regressão.


In [ ]:
df_modelagem = df.copy()

print(f"Registros antes da análise de outliers: {len(df)}")
print(f"Registros mantidos para as próximas etapas: {len(df_modelagem)}")
print("✅ Nenhuma observação removida.")


## ✅ Conclusão parcial — Outliers

A análise demonstrou que a identificação de outliers não deve ser feita de maneira automática.

O principal achado foi que uma análise global de `Yield` classificaria incorretamente dezenas de observações como discrepantes apenas porque diferentes culturas possuem escalas de rendimento distintas.

Ao considerar o contexto de cada cultura, não foram encontrados outliers de rendimento que justificassem remoção.

Assim, a base foi preservada integralmente para a próxima etapa.

## Próxima etapa

A próxima seção será dedicada à **clusterização**, com o objetivo de identificar grupos naturais de observações e tendências de produtividade sem utilizar previamente os rótulos de rendimento como resposta supervisionada.


# 7. Clusterização — identificação de padrões ambientais

Nesta etapa será utilizada uma técnica de Machine Learning **não supervisionada** para identificar grupos naturais de observações com características ambientais semelhantes.

A clusterização será feita sem utilizar `Yield` como variável de entrada.

Essa decisão é importante porque queremos descobrir os padrões ambientais existentes na base sem informar ao algoritmo qual foi o rendimento observado.

Também não utilizaremos `Crop` na formação dos clusters, pois o objetivo desta etapa é identificar **cenários ambientais**, e não simplesmente separar as culturas.

As variáveis utilizadas serão:

- precipitação;
- umidade específica;
- umidade relativa;
- temperatura.


## 7.1 Preparação dos dados para clusterização

O algoritmo K-Means é sensível à escala das variáveis.

Por exemplo, a precipitação possui valores na casa dos milhares, enquanto a temperatura está próxima de 26 °C.

Se utilizássemos os valores originais, a precipitação poderia exercer influência excessiva apenas por possuir uma escala numérica maior.

Por isso, as variáveis serão padronizadas com `StandardScaler`.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features_cluster = [
    "Precipitation (mm day-1)",
    "Specific Humidity at 2 Meters (g/kg)",
    "Relative Humidity at 2 Meters (%)",
    "Temperature at 2 Meters (C)",
]

# Existem 39 perfis ambientais únicos, repetidos para cada uma das 4 culturas.
# Para que cada cenário ambiental tenha peso apenas uma vez na formação dos
# clusters, a clusterização será ajustada sobre os perfis únicos.
perfis_unicos = (
    df_modelagem[features_cluster]
    .drop_duplicates()
    .reset_index(drop=True)
)

scaler_cluster = StandardScaler()

X_cluster = scaler_cluster.fit_transform(perfis_unicos)

print(f"Perfis ambientais usados na clusterização: {len(perfis_unicos)}")
print(f"Quantidade de variáveis: {len(features_cluster)}")


## 7.2 Escolha do número de clusters

Não é recomendável escolher arbitrariamente o valor de `k`.

Serão utilizados dois critérios:

1. **Método do Cotovelo (Elbow Method)** — observa a redução da inércia conforme o número de clusters aumenta;
2. **Silhouette Score** — mede o grau de separação entre os grupos, variando aproximadamente de -1 a 1.

Quanto maior o Silhouette Score, melhor tende a ser a separação dos clusters.


In [ ]:
resultados_k = []

for k in range(2, 9):
    modelo_k = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=20
    )

    labels_k = modelo_k.fit_predict(X_cluster)

    resultados_k.append({
        "k": k,
        "Inércia": modelo_k.inertia_,
        "Silhouette": silhouette_score(X_cluster, labels_k)
    })

resultados_k = pd.DataFrame(resultados_k)

display(resultados_k.round(4))


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    resultados_k["k"],
    resultados_k["Inércia"],
    marker="o"
)

plt.title("Método do Cotovelo — K-Means")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inércia")
plt.xticks(resultados_k["k"])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    resultados_k["k"],
    resultados_k["Silhouette"],
    marker="o"
)

plt.title("Silhouette Score por Número de Clusters")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette Score")
plt.xticks(resultados_k["k"])
plt.tight_layout()
plt.show()


### Interpretação da escolha de `k`

Na base analisada, o maior Silhouette Score ocorre com **3 clusters**, com valor próximo de **0,40**.

O método do cotovelo também apresenta uma redução relevante da inércia até aproximadamente essa região.

Por isso, será adotado:

**k = 3**

Essa escolha fornece uma segmentação simples o suficiente para interpretação e, ao mesmo tempo, representa diferentes cenários ambientais encontrados na base.


## 7.3 Treinamento do K-Means

Agora treinamos o modelo definitivo com três clusters.

Depois, o cluster de cada perfil ambiental será associado novamente aos 156 registros originais, permitindo analisar o comportamento do rendimento em cada grupo.


In [ ]:
K_ESCOLHIDO = 3

kmeans = KMeans(
    n_clusters=K_ESCOLHIDO,
    random_state=RANDOM_STATE,
    n_init=20
)

perfis_unicos["Cluster"] = kmeans.fit_predict(X_cluster)

df_clusterizado = df_modelagem.merge(
    perfis_unicos,
    on=features_cluster,
    how="left"
)

print("Quantidade de registros por cluster:")
display(
    df_clusterizado["Cluster"]
    .value_counts()
    .sort_index()
    .rename("Quantidade")
    .to_frame()
)


## 7.4 Perfil ambiental dos clusters

Para interpretar os grupos, calculamos as médias das variáveis ambientais em cada cluster.


In [ ]:
perfil_clusters = (
    df_clusterizado
    .groupby("Cluster")[features_cluster]
    .mean()
    .round(3)
)

display(perfil_clusters)


### Interpretação dos perfis

Os três clusters representam cenários ambientais diferentes.

De forma geral:

- um grupo concentra condições com **maior precipitação**;
- outro apresenta **temperaturas médias mais elevadas**;
- outro possui **menores níveis médios de precipitação e temperatura**.

Os clusters não devem ser interpretados como categorias agronômicas absolutas, mas como agrupamentos estatísticos formados a partir das condições presentes na base.


## 7.5 Tendência de rendimento por cluster

Depois de formar os grupos usando apenas variáveis ambientais, analisamos o rendimento observado dentro de cada cluster.

Essa etapa permite verificar se diferentes cenários ambientais estão associados a diferentes níveis de produtividade.


In [ ]:
rendimento_cluster = (
    df_clusterizado
    .groupby("Cluster")["Yield"]
    .agg(
        quantidade="count",
        media="mean",
        mediana="median",
        minimo="min",
        maximo="max"
    )
    .round(2)
)

display(rendimento_cluster)


In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df_clusterizado,
    x="Cluster",
    y="Yield"
)

plt.title("Distribuição de Yield por Cluster Ambiental")
plt.xlabel("Cluster")
plt.ylabel("Yield")
plt.tight_layout()
plt.show()


### Interpretação

Os rendimentos médios globais dos clusters são relativamente próximos quando todas as culturas são misturadas.

Isso ocorre porque cada perfil ambiental aparece para todas as quatro culturas e a variável `Crop` exerce forte influência sobre a escala do rendimento.

Portanto, para entender melhor a tendência de produtividade, também é necessário observar a combinação **cluster × cultura**.


In [ ]:
rendimento_cluster_cultura = (
    df_clusterizado
    .groupby(["Cluster", "Crop"])["Yield"]
    .mean()
    .unstack()
    .round(2)
)

display(rendimento_cluster_cultura)


In [ ]:
plt.figure(figsize=(11, 6))

sns.barplot(
    data=df_clusterizado,
    x="Cluster",
    y="Yield",
    hue="Crop",
    errorbar=None
)

plt.title("Rendimento Médio por Cluster e Cultura")
plt.xlabel("Cluster")
plt.ylabel("Yield médio")
plt.tight_layout()
plt.show()


### Conclusão da clusterização

A clusterização confirmou que existem diferentes cenários ambientais na base.

Entretanto, o efeito desses cenários sobre o rendimento varia dependendo da cultura.

Isso reforça uma conclusão já observada na EDA: **a cultura agrícola precisa ser considerada nos modelos supervisionados**, pois o mesmo cenário ambiental pode produzir rendimentos muito diferentes para culturas distintas.


# 8. Preparação para os modelos de regressão

A partir desta seção inicia-se a etapa supervisionada.

O objetivo é prever:

**`Yield`**

As variáveis utilizadas como entrada serão:

- `Crop`;
- precipitação;
- umidade específica;
- umidade relativa;
- temperatura.

Como `Crop` é uma variável categórica, ela será convertida para representação numérica usando **One-Hot Encoding**.

As variáveis numéricas serão padronizadas dentro de um `Pipeline`, evitando que o pré-processamento seja realizado de forma diferente entre treino e teste.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df_modelagem.drop(columns=["Yield"])
y = df_modelagem["Yield"]

colunas_categoricas = ["Crop"]

colunas_numericas_modelo = [
    "Precipitation (mm day-1)",
    "Specific Humidity at 2 Meters (g/kg)",
    "Relative Humidity at 2 Meters (%)",
    "Temperature at 2 Meters (C)",
]

preprocessador = ColumnTransformer(
    transformers=[
        (
            "categorica",
            OneHotEncoder(handle_unknown="ignore"),
            colunas_categoricas
        ),
        (
            "numerica",
            StandardScaler(),
            colunas_numericas_modelo
        )
    ]
)

print(f"Variáveis de entrada: {X.shape[1]}")
print(f"Registros disponíveis: {X.shape[0]}")


## 8.1 Separação entre treino e teste

A base será dividida em:

- **80% para treinamento**;
- **20% para teste**.

A divisão utiliza `random_state=42` para garantir reprodutibilidade.

Também será utilizada estratificação pela cultura para manter uma distribuição semelhante das quatro culturas nos conjuntos de treino e teste.

O conjunto de teste permanecerá separado durante o treinamento e será utilizado apenas para avaliar a capacidade de generalização dos modelos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=X["Crop"]
)

print(f"Treino: {len(X_train)} registros")
print(f"Teste:  {len(X_test)} registros")

print("\nDistribuição das culturas no treino:")
display(X_train["Crop"].value_counts().sort_index().to_frame("Quantidade"))

print("Distribuição das culturas no teste:")
display(X_test["Crop"].value_counts().sort_index().to_frame("Quantidade"))


# 9. Métricas de avaliação

Como o problema é de regressão, serão utilizadas as seguintes métricas:

### MAE — Mean Absolute Error

Representa o erro absoluto médio das previsões.

Quanto menor, melhor.

### RMSE — Root Mean Squared Error

Penaliza erros grandes com maior intensidade.

Quanto menor, melhor.

### R² — Coeficiente de determinação

Indica quanto da variação do alvo é explicada pelo modelo.

Valores mais próximos de 1 indicam melhor ajuste.

Nenhuma métrica será analisada isoladamente. A comparação final considerará o conjunto dos resultados.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def avaliar_regressao(nome_modelo, y_real, y_previsto):
    mae = mean_absolute_error(y_real, y_previsto)
    rmse = np.sqrt(mean_squared_error(y_real, y_previsto))
    r2 = r2_score(y_real, y_previsto)

    return {
        "Modelo": nome_modelo,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    }


# 10. Modelos de regressão

Nesta versão do notebook serão implementados **três dos cinco modelos** exigidos no projeto.

Os três modelos escolhidos são:

1. Regressão Linear;
2. Random Forest Regressor;
3. Gradient Boosting Regressor.

Os **Modelos 4 e 5 serão adicionados posteriormente por outro integrante do grupo**, utilizando algoritmos diferentes.

Essa separação evita duplicidade de trabalho entre os integrantes.


## 10.1 Modelo 1 — Regressão Linear

A Regressão Linear será utilizada como modelo de referência (*baseline*).

Ela tenta representar `Yield` como uma combinação linear das variáveis de entrada.

Mesmo quando existem relações complexas, um modelo simples é importante porque fornece um ponto inicial para comparar se algoritmos mais sofisticados realmente oferecem ganho de desempenho.


In [ ]:
from sklearn.linear_model import LinearRegression

pipeline_linear = Pipeline(
    steps=[
        ("preprocessamento", preprocessador),
        ("modelo", LinearRegression())
    ]
)

pipeline_linear.fit(X_train, y_train)

pred_linear = pipeline_linear.predict(X_test)

resultado_linear = avaliar_regressao(
    "Regressão Linear",
    y_test,
    pred_linear
)

display(pd.DataFrame([resultado_linear]).round(4))


## 10.2 Modelo 2 — Random Forest Regressor

O Random Forest combina diversas árvores de decisão.

Ele é capaz de representar relações não lineares e interações entre as variáveis sem exigir uma fórmula matemática previamente definida.

Sua utilização é interessante neste problema porque a resposta das culturas às condições ambientais pode não seguir uma relação puramente linear.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

pipeline_rf = Pipeline(
    steps=[
        ("preprocessamento", preprocessador),
        (
            "modelo",
            RandomForestRegressor(
                n_estimators=400,
                random_state=RANDOM_STATE
            )
        )
    ]
)

pipeline_rf.fit(X_train, y_train)

pred_rf = pipeline_rf.predict(X_test)

resultado_rf = avaliar_regressao(
    "Random Forest",
    y_test,
    pred_rf
)

display(pd.DataFrame([resultado_rf]).round(4))


## 10.3 Modelo 3 — Gradient Boosting Regressor

O Gradient Boosting cria uma sequência de árvores na qual cada nova etapa tenta corrigir os erros cometidos anteriormente.

É um algoritmo de ensemble diferente do Random Forest e pode capturar relações complexas entre as variáveis.

Ele será utilizado como terceiro modelo para aumentar a diversidade da comparação.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

pipeline_gb = Pipeline(
    steps=[
        ("preprocessamento", preprocessador),
        (
            "modelo",
            GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=2,
                random_state=RANDOM_STATE
            )
        )
    ]
)

pipeline_gb.fit(X_train, y_train)

pred_gb = pipeline_gb.predict(X_test)

resultado_gb = avaliar_regressao(
    "Gradient Boosting",
    y_test,
    pred_gb
)

display(pd.DataFrame([resultado_gb]).round(4))


## 10.4 Modelos 4 e 5 — pendentes

Os dois modelos abaixo serão escolhidos e implementados por outro integrante:

- **Modelo 4:** pendente;
- **Modelo 5:** pendente.

Eles deverão seguir exatamente o mesmo padrão:

1. criar um `Pipeline`;
2. treinar somente em `X_train` e `y_train`;
3. prever `X_test`;
4. calcular MAE, RMSE e R²;
5. adicionar os resultados à tabela comparativa.

> Na versão final entregue à FIAP, a comparação deverá conter os **cinco algoritmos**, conforme solicitado no enunciado.


# 11. Comparação parcial dos três modelos

Enquanto os outros dois modelos ainda não foram adicionados, podemos comparar os três algoritmos já implementados.

Esta tabela é **parcial** e deverá ser atualizada quando os Modelos 4 e 5 forem concluídos.


In [ ]:
resultados_modelos = pd.DataFrame([
    resultado_linear,
    resultado_rf,
    resultado_gb
])

resultados_modelos = (
    resultados_modelos
    .sort_values("R²", ascending=False)
    .reset_index(drop=True)
)

display(resultados_modelos.round(4))


In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=resultados_modelos,
    x="Modelo",
    y="R²"
)

plt.title("Comparação Parcial — R² dos Modelos")
plt.xlabel("Modelo")
plt.ylabel("R²")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
comparacao_erros = resultados_modelos.melt(
    id_vars="Modelo",
    value_vars=["MAE", "RMSE"],
    var_name="Métrica",
    value_name="Erro"
)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=comparacao_erros,
    x="Modelo",
    y="Erro",
    hue="Métrica"
)

plt.title("Comparação Parcial — MAE e RMSE")
plt.xlabel("Modelo")
plt.ylabel("Erro")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### Interpretação parcial

Os três modelos apresentam desempenho elevado no conjunto de teste.

Entretanto, esse resultado deve ser interpretado com cuidado.

A EDA mostrou que a variável `Crop` está fortemente associada à escala do rendimento. Como as quatro culturas apresentam níveis médios muito diferentes de `Yield`, identificar corretamente a cultura já fornece grande quantidade de informação ao modelo.

Portanto, um R² elevado não deve ser interpretado como prova de que as condições ambientais explicam sozinhas quase todo o rendimento.

A análise final deverá considerar também:

- validação cruzada;
- estabilidade entre diferentes divisões dos dados;
- desempenho dos dois modelos ainda pendentes;
- características estruturais da base.


# 12. Validação cruzada

Uma única divisão treino/teste pode favorecer ou prejudicar um modelo dependendo das observações escolhidas.

Por isso, utilizamos **validação cruzada com 5 folds**.

A base é dividida em cinco partes. Em cada rodada:

- quatro partes são utilizadas para treinamento;
- uma parte é utilizada para validação.

O processo é repetido cinco vezes.

Isso fornece uma avaliação mais robusta da estabilidade dos modelos.


In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

modelos_cv = {
    "Regressão Linear": pipeline_linear,
    "Random Forest": pipeline_rf,
    "Gradient Boosting": pipeline_gb,
}

resultados_cv = []

for nome, modelo in modelos_cv.items():

    scores = cross_validate(
        modelo,
        X,
        y,
        cv=cv,
        scoring={
            "r2": "r2",
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error"
        }
    )

    resultados_cv.append({
        "Modelo": nome,
        "R² médio": scores["test_r2"].mean(),
        "Desvio R²": scores["test_r2"].std(),
        "MAE médio": -scores["test_mae"].mean(),
        "RMSE médio": np.sqrt(-scores["test_mse"]).mean()
    })

resultados_cv = (
    pd.DataFrame(resultados_cv)
    .sort_values("R² médio", ascending=False)
    .reset_index(drop=True)
)

display(resultados_cv.round(4))


### Interpretação da validação cruzada

A validação cruzada permite verificar se o bom desempenho observado no teste inicial se mantém em diferentes divisões da base.

Na versão atual, os três algoritmos continuam apresentando R² médio elevado.

A diferença entre eles é relativamente pequena, o que significa que a escolha final não deve considerar apenas um único número.

Quando os dois modelos restantes forem adicionados, eles também deverão passar pela mesma validação cruzada antes da escolha definitiva.


# 13. Valores reais versus valores previstos

Um gráfico de valores reais contra valores previstos permite avaliar visualmente o comportamento dos modelos.

Quanto mais próximos os pontos estiverem da linha diagonal, mais próximas as previsões estão dos valores reais.


In [ ]:
previsoes_teste = pd.DataFrame({
    "Real": y_test.values,
    "Regressão Linear": pred_linear,
    "Random Forest": pred_rf,
    "Gradient Boosting": pred_gb
})

for modelo in ["Regressão Linear", "Random Forest", "Gradient Boosting"]:

    plt.figure(figsize=(7, 6))

    plt.scatter(
        previsoes_teste["Real"],
        previsoes_teste[modelo],
        alpha=0.75
    )

    minimo = min(
        previsoes_teste["Real"].min(),
        previsoes_teste[modelo].min()
    )

    maximo = max(
        previsoes_teste["Real"].max(),
        previsoes_teste[modelo].max()
    )

    plt.plot(
        [minimo, maximo],
        [minimo, maximo],
        linestyle="--"
    )

    plt.title(f"Valores Reais × Previstos — {modelo}")
    plt.xlabel("Yield real")
    plt.ylabel("Yield previsto")
    plt.tight_layout()
    plt.show()


# 14. Importância das variáveis — Random Forest

Modelos baseados em árvores permitem estimar a importância relativa das variáveis utilizadas nas decisões.

Essa análise não representa causalidade, mas ajuda a compreender quais características o modelo utilizou com maior intensidade.


In [ ]:
nomes_features_rf = (
    pipeline_rf
    .named_steps["preprocessamento"]
    .get_feature_names_out()
)

importancias_rf = (
    pipeline_rf
    .named_steps["modelo"]
    .feature_importances_
)

tabela_importancias = (
    pd.DataFrame({
        "Variável": nomes_features_rf,
        "Importância": importancias_rf
    })
    .sort_values("Importância", ascending=False)
    .reset_index(drop=True)
)

display(tabela_importancias.round(4))


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=tabela_importancias,
    x="Importância",
    y="Variável"
)

plt.title("Importância das Variáveis — Random Forest")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.tight_layout()
plt.show()


### Interpretação da importância das variáveis

A variável associada à cultura `Oil palm fruit` tende a apresentar importância muito superior às demais.

Isso é coerente com a EDA, pois essa cultura possui uma escala de rendimento muito maior que os outros grupos.

As variáveis ambientais aparecem com importância menor no Random Forest global.

Esse resultado reforça novamente que:

**o tipo de cultura é uma informação central para a previsão de `Yield` nesta base.**

Isso não significa que as variáveis ambientais sejam irrelevantes. As análises por cultura mostraram que algumas delas apresentam relações relevantes dentro de grupos específicos.


# 15. Limitações da análise

Apesar dos bons resultados obtidos, algumas limitações precisam ser consideradas.

### 1. Tamanho da base

O dataset possui apenas **156 observações**.

Bases pequenas aumentam a incerteza sobre a capacidade de generalização para cenários agrícolas muito diferentes dos observados.

### 2. Apenas 39 perfis ambientais únicos

As quatro culturas compartilham os mesmos 39 conjuntos de condições ambientais.

Isso é útil para comparação entre culturas, mas reduz a diversidade ambiental efetiva da base.

### 3. Forte influência da cultura

As culturas apresentam escalas de rendimento muito diferentes.

Por isso, a variável `Crop` explica grande parte das diferenças observadas no alvo.

### 4. Variáveis agrícolas ausentes

Outros fatores que podem influenciar rendimento não estão presentes, como:

- características do solo;
- fertilização;
- irrigação;
- ocorrência de pragas;
- manejo;
- variedade genética;
- radiação solar.

### 5. Associação não significa causalidade

Correlação, clusterização e importância de variáveis mostram padrões estatísticos, mas não provam relações causais.

Essas limitações devem ser levadas em consideração ao interpretar os resultados.


# 16. Conclusão geral — versão parcial com 3 modelos

O projeto realizou as principais etapas de um fluxo de Machine Learning aplicado à previsão de rendimento agrícola:

1. carregamento e validação da base;
2. análise de qualidade;
3. estatística descritiva;
4. EDA;
5. investigação de outliers;
6. clusterização;
7. preparação das variáveis;
8. divisão treino/teste;
9. treinamento de três modelos de regressão;
10. avaliação com MAE, RMSE e R²;
11. validação cruzada;
12. análise de valores reais versus previstos;
13. análise de importância das variáveis.

A clusterização identificou **três cenários ambientais**, mostrando que diferentes combinações de precipitação, umidade e temperatura podem ser agrupadas em perfis semelhantes.

Na modelagem supervisionada, os três algoritmos implementados apresentaram forte capacidade de ajuste à base analisada.

Entretanto, os resultados também mostraram que a variável `Crop` possui papel decisivo, pois as culturas apresentam escalas de rendimento muito diferentes.

## Situação da entrega

Nesta versão estão prontos:

- **Modelo 1 — Regressão Linear**
- **Modelo 2 — Random Forest Regressor**
- **Modelo 3 — Gradient Boosting Regressor**

Ainda devem ser acrescentados pelo outro integrante:

- **Modelo 4 — pendente**
- **Modelo 5 — pendente**

Somente após a implementação e avaliação dos cinco algoritmos deverá ser escrita a **conclusão definitiva sobre o melhor modelo**.


# 17. Checklist antes da entrega final

Antes de entregar o notebook, o grupo deverá confirmar:

- [x] análise exploratória;
- [x] análise de outliers;
- [x] clusterização;
- [x] justificativa para escolha de `k`;
- [x] três modelos de regressão;
- [x] MAE;
- [x] RMSE;
- [x] R²;
- [x] validação cruzada;
- [x] gráficos de comparação;
- [x] análise de limitações;
- [ ] Modelo 4;
- [ ] Modelo 5;
- [ ] atualizar comparação com os cinco modelos;
- [ ] escolher o melhor modelo final;
- [ ] atualizar a conclusão definitiva;
- [ ] executar todas as células antes da entrega;
- [ ] conferir links do README e do vídeo.
